# Connecting to the ML Ecosystem

**polars-ts** is designed as a *data-preparation layer* that sits between raw time-series
data and the deep-learning / reinforcement-learning frameworks you already use.

This notebook walks through the adapter functions that convert polars-ts
DataFrames into the formats expected by:

| Adapter | Target framework | Direction |
|---|---|---|
| `to_neuralforecast` / `from_neuralforecast` | [Nixtla NeuralForecast](https://nixtlaverse.nixtla.io/neuralforecast/) | round-trip |
| `to_pytorch_forecasting` / `from_pytorch_forecasting` | [PyTorch Forecasting](https://pytorch-forecasting.readthedocs.io/) | round-trip |
| `to_hf_dataset` | [HuggingFace Datasets](https://huggingface.co/docs/datasets/) | export |
| `ForecastEnv` | RL environments (Gymnasium-style) | interactive |
| `to_chronos_embeddings` | [Amazon Chronos](https://github.com/amazon-science/chronos-forecasting) | export |
| `to_moment_embeddings` | [CMU MOMENT](https://github.com/moment-timeseries-foundation-model/moment) | export |

The goal is always the same: **prepare once in polars-ts, train anywhere**.

## 1. Imports & data loading

We use two classic datasets:
* **Air Passengers** — a single monthly series (144 observations)
* **M4 Hourly (H1)** — a single hourly series from the M4 competition

In [ ]:
import numpy as np
import polars as pl

from polars_ts import (
    ForecastEnv,
    from_neuralforecast,
    from_pytorch_forecasting,
    naive_forecast,
    to_hf_dataset,
    to_neuralforecast,
    to_pytorch_forecasting,
    to_chronos_embeddings,
    to_moment_embeddings,
)

In [ ]:
# Air Passengers — monthly, single series
air = pl.read_csv(
    "https://datasets-nixtla.s3.amazonaws.com/air-passengers.csv",
    try_parse_dates=True,
).with_columns(pl.lit("AP").alias("unique_id"))

print(f"Air Passengers: {air.shape}")
air.head()

In [ ]:
# M4 Hourly — single series H1
m4 = (
    pl.scan_parquet("https://datasets-nixtla.s3.amazonaws.com/m4-hourly.parquet")
    .filter(pl.col("unique_id") == "H1")
    .collect()
)

print(f"M4 Hourly H1: {m4.shape}")
m4.head()

## 2. NeuralForecast adapter

[NeuralForecast](https://nixtlaverse.nixtla.io/neuralforecast/) expects a DataFrame
with exactly three columns: **unique_id**, **ds** (datetime), and **y** (target).

`to_neuralforecast` renames your columns to match that convention;
`from_neuralforecast` converts a NeuralForecast result back into polars-ts format.

In [ ]:
# Convert Air Passengers to NeuralForecast format
nf_df = to_neuralforecast(air)

print("NeuralForecast format:")
print(f"  Columns: {nf_df.columns}")
print(f"  Shape:   {nf_df.shape}")
nf_df.head()

In [ ]:
# Round-trip: simulate a NeuralForecast result and convert back
# NeuralForecast returns a DataFrame with model-named columns alongside unique_id / ds
mock_result = (
    nf_df.tail(12)
    .with_columns(
        pl.col("y").alias("NHITS"),  # mock forecast column
    )
    .drop("y")
)

print("Mock NeuralForecast result:")
print(mock_result.head())

# Convert back
back = from_neuralforecast(mock_result)
print("\nConverted back to polars-ts format:")
back.head()

In [ ]:
# Optional: fit a real NeuralForecast model if the library is installed
try:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NHITS

    nf = NeuralForecast(models=[NHITS(h=12, input_size=24, max_steps=50)], freq="MS")
    nf.fit(df=nf_df.to_pandas())
    preds = pl.from_pandas(nf.predict())
    result = from_neuralforecast(preds)
    print(result)
except ImportError:
    print("neuralforecast not installed — skipping live model fitting.")
    print("Install with: pip install neuralforecast")

## 3. PyTorch Forecasting adapter

[PyTorch Forecasting](https://pytorch-forecasting.readthedocs.io/) needs a *pandas*
DataFrame with an integer time index and group identifiers.

`to_pytorch_forecasting` returns a **dict** with two keys:
* `"data"` — a pandas DataFrame ready for `TimeSeriesDataSet`
* `"metadata"` — a dict describing column roles (`time_idx`, `group_ids`, `target`, etc.)

In [ ]:
ptf = to_pytorch_forecasting(air)

print("Keys:", list(ptf.keys()))
print("\nMetadata:")
for k, v in ptf["metadata"].items():
    print(f"  {k}: {v}")

print(f"\nData shape: {ptf['data'].shape}")
ptf["data"].head()

In [ ]:
# Simulate a PyTorch Forecasting prediction array and convert back
import pandas as pd

mock_preds = pd.DataFrame(
    {
        "unique_id": ["AP"] * 12,
        "ds": pd.date_range("1961-01", periods=12, freq="MS"),
        "prediction": np.random.default_rng(42).normal(400, 50, size=12),
    }
)

back_ptf = from_pytorch_forecasting(mock_preds)
print("Converted back:")
back_ptf.head()

## 4. HuggingFace Datasets adapter

The [HuggingFace time-series format](https://huggingface.co/docs/datasets/) stores each
series as a single row with columns:

| Column | Description |
|---|---|
| `id` | series identifier (string) |
| `target` | list of float values |
| `start` | timestamp of the first observation |

`to_hf_dataset` reshapes a long-format polars-ts DataFrame into this wide format.

In [ ]:
try:
    hf_ds = to_hf_dataset(air)
    print(f"Type: {type(hf_ds)}")
    print(f"Features: {hf_ds.features}")
    print(f"Num rows: {hf_ds.num_rows}")
    print()
    print("First row:")
    row = hf_ds[0]
    print(f"  id:     {row['id']}")
    print(f"  start:  {row['start']}")
    print(f"  target: {row['target'][:5]} ... ({len(row['target'])} values)")
except ImportError:
    print("datasets library not installed — skipping.")
    print("Install with: pip install datasets")

## 5. ForecastEnv — RL environment

`ForecastEnv` wraps actuals and forecasts into a Gymnasium-style environment.
At each step the agent observes a sliding window of recent data and chooses an
action (e.g., an adjusted forecast). The reward reflects forecast accuracy.

This is useful for training RL agents that learn to *correct* base forecasts.

In [ ]:
# Generate naive forecasts for the M4 H1 series
fc = naive_forecast(m4, h=50)

actuals = m4.tail(50)["y"].to_numpy()
forecasts = fc["y_hat"].to_numpy()

print(f"Actuals shape:   {actuals.shape}")
print(f"Forecasts shape: {forecasts.shape}")
print(f"First 5 actuals:   {actuals[:5]}")
print(f"First 5 forecasts: {forecasts[:5]}")

In [ ]:
# Create the RL environment and run a simple episode
env = ForecastEnv(data=actuals, forecasts=forecasts, window_size=10)
obs = env.reset()
print(f"Initial observation shape: {obs.shape}")
print(f"Initial observation: {obs}")

total_reward = 0.0
done = False
steps = 0

while not done:
    # Simple strategy: use the last forecast value as our action
    action = obs[-1]
    obs, reward, done, info = env.step(action)
    total_reward += reward
    steps += 1

print(f"\nEpisode finished after {steps} steps")
print(f"Total reward: {total_reward:.2f}")

## 6. Foundation model embeddings

Foundation models like **Chronos** (Amazon) and **MOMENT** (CMU) are pre-trained
on millions of time series. Rather than fine-tuning them for forecasting, we can
extract their **encoder embeddings** and use them as feature vectors for downstream
tasks like clustering, classification, or anomaly detection.

polars-ts provides two adapters:

| Adapter | Model | Output |
|---|---|---|
| `to_chronos_embeddings` | Chronos-T5 (small/base/large) | `[unique_id, emb_0, ..., emb_d]` |
| `to_moment_embeddings` | MOMENT-1 (large) | `[unique_id, emb_0, ..., emb_d]` |

Both require `torch`. Chronos additionally needs `transformers`; MOMENT needs `momentfm`.

In [ ]:
# Chronos embeddings — extract from the T5 encoder
try:
    chronos_emb = to_chronos_embeddings(
        m4,
        model_name="amazon/chronos-t5-small",
        device="cpu",
    )
    print(f"Chronos embedding shape: {chronos_emb.shape}")
    print(f"Columns: {chronos_emb.columns[:5]} ... ({len(chronos_emb.columns)} total)")
    chronos_emb.head()
except ImportError as e:
    print(f"Skipping Chronos: {e}")
    print("Install with: pip install torch transformers")

In [ ]:
# MOMENT embeddings — extract from the MOMENT encoder
try:
    moment_emb = to_moment_embeddings(
        m4,
        model_name="AutonLab/MOMENT-1-large",
        device="cpu",
    )
    print(f"MOMENT embedding shape: {moment_emb.shape}")
    print(f"Columns: {moment_emb.columns[:5]} ... ({len(moment_emb.columns)} total)")
    moment_emb.head()
except ImportError as e:
    print(f"Skipping MOMENT: {e}")
    print("Install with: pip install torch momentfm")

### Embedding → clustering workflow

Once we have embeddings, we can cluster them with any standard algorithm.
Below we use embeddings from multiple M4 series and cluster with scikit-learn KMeans.

In [ ]:
# Load 20 M4 series and extract embeddings
m4_multi = (
    pl.scan_parquet("https://datasets-nixtla.s3.amazonaws.com/m4-hourly.parquet")
    .filter(pl.col("unique_id").str.strip_chars_start("H").cast(pl.Int64) <= 20)
    .collect()
)

try:
    emb = to_chronos_embeddings(m4_multi, device="cpu")
    print(f"Embeddings: {emb.shape[0]} series x {emb.shape[1] - 1} dimensions")

    # Cluster with KMeans
    from sklearn.cluster import KMeans

    X = emb.drop("unique_id").to_numpy()
    km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)

    labels = emb.select("unique_id").with_columns(pl.Series("cluster", km.labels_))
    print("\nCluster assignments:")
    print(labels)
except ImportError as e:
    print(f"Skipping: {e}")

In [ ]:
# t-SNE visualisation of embeddings coloured by cluster
try:
    from sklearn.manifold import TSNE

    tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, X.shape[0] - 1))
    coords = tsne.fit_transform(X)

    viz_df = pl.DataFrame(
        {
            "tsne_1": coords[:, 0],
            "tsne_2": coords[:, 1],
            "cluster": [str(c) for c in km.labels_],
            "unique_id": emb["unique_id"].to_list(),
        }
    )

    import hvplot.polars  # noqa

    viz_df.hvplot.scatter(
        x="tsne_1",
        y="tsne_2",
        by="cluster",
        hover_cols=["unique_id"],
        width=600,
        height=400,
        title="Chronos Embeddings — t-SNE (coloured by KMeans cluster)",
        size=100,
    )
except Exception as e:
    print(f"Skipping t-SNE visualisation: {e}")

## Summary

| Adapter | Use when you want to... |
|---|---|
| `to_neuralforecast` | Train NHITS, PatchTST, or any Nixtla neural model |
| `to_pytorch_forecasting` | Build TFT or DeepAR models with PyTorch Forecasting |
| `to_hf_dataset` | Upload series to the HuggingFace Hub or use HF pipelines |
| `ForecastEnv` | Train an RL agent to improve base forecasts online |
| `to_chronos_embeddings` | Extract Chronos encoder embeddings for clustering/classification |
| `to_moment_embeddings` | Extract MOMENT encoder embeddings for clustering/classification |

All adapters work with standard polars-ts long-format DataFrames
(`unique_id`, `ds`, `y` + optional columns), so your data-prep pipeline
stays the same regardless of the downstream framework.

### Further reading

* [NeuralForecast docs](https://nixtlaverse.nixtla.io/neuralforecast/)
* [PyTorch Forecasting docs](https://pytorch-forecasting.readthedocs.io/)
* [HuggingFace Datasets — Time Series](https://huggingface.co/docs/datasets/)
* [Gymnasium (RL environments)](https://gymnasium.farama.org/)
* Jansen, S. *Modern Time Series Forecasting with Python*, 2nd Ed. — Ch. 15–17